# CIC-IDS2017 Data-Cleaning Audit (No Model Training)

This audit reproduces the data-loading, invalid-row removal, constant-column removal, duplicate-removal, binary-label mapping, and split-size checks used in the AE–GNN–RF experiment. **It does not train the Autoencoder, GCN, or Random Forest.**

## Before running
Place this notebook in the same folder as the eight CSV files listed in the first code cell. Run all cells from top to bottom. The audit saves CSV reports so the counts can be checked and used to correct the manuscript.

**Important:** the audit intentionally reports both (a) the number of columns remaining after constant-column removal and (b) the number of numeric columns actually passed through the numeric-only filtering step. These numbers may differ; do not assume they are identical.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

_CWD = Path.cwd().resolve()
if (_CWD / "README.md").exists():
    PROJECT_ROOT = _CWD
elif (_CWD.parent / "README.md").exists():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

candidates = [PROJECT_ROOT / "data" / "raw" / "cicids2017", Path.cwd(), PROJECT_ROOT, Path("/content")]
DATA_DIR = next((d for d in candidates if all((d / name).exists() for name in files)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Put the eight CIC-IDS2017 CSV files in data/raw/cicids2017/ as explained in README.md.")
TARGET_COL = "Label"
SEED = 42
print("Using dataset directory:", DATA_DIR)


In [ ]:
# ============================================================
# 1. Load each source file and audit invalid rows individually
# ============================================================
dfs = []
file_audit_rows = []

for file in FILES:
    df = pd.read_csv(DATA_DIR / file, encoding='latin1', low_memory=False)
    df.columns = df.columns.str.strip()
    if TARGET_COL not in df.columns:
        raise KeyError(f"'{TARGET_COL}' was not found in {file}. Available columns: {list(df.columns)}")

    raw_rows, raw_cols = df.shape
    missing_row = df.isna().any(axis=1)
    numeric = df.select_dtypes(include=[np.number])
    if numeric.shape[1]:
        inf_row = pd.Series(np.isinf(numeric.to_numpy(dtype=np.float64, copy=False)).any(axis=1), index=df.index)
        inf_cell_count = int(np.isinf(numeric.to_numpy(dtype=np.float64, copy=False)).sum())
    else:
        inf_row = pd.Series(False, index=df.index)
        inf_cell_count = 0

    invalid_row = missing_row | inf_row
    cleaned = df.replace([np.inf, -np.inf], np.nan).dropna()
    label_values = cleaned[TARGET_COL].astype(str)

    file_audit_rows.append({
        'File': file,
        'Raw_rows': raw_rows,
        'Raw_columns': raw_cols,
        'Missing_cells_before_cleaning': int(df.isna().sum().sum()),
        'Infinite_numeric_cells_before_cleaning': inf_cell_count,
        'Rows_with_any_missing_value': int(missing_row.sum()),
        'Rows_with_any_infinity': int(inf_row.sum()),
        'Rows_with_both_missing_and_infinity': int((missing_row & inf_row).sum()),
        'Rows_removed_by_replace_inf_then_dropna': int(invalid_row.sum()),
        'Rows_retained_after_current_cleaning': int(len(cleaned)),
        'Clean_BENIGN_exact_label_rows': int((label_values == 'BENIGN').sum()),
        'Clean_non_BENIGN_exact_label_rows': int((label_values != 'BENIGN').sum()),
    })

    # Preserve the unmodified source dataframe (apart from stripped column names),
    # so the combined cleaning below follows the original notebook sequence.
    dfs.append(df)
    print(f'{file}: {raw_rows:,} raw rows -> {len(cleaned):,} rows after current dropna rule')

file_audit = pd.DataFrame(file_audit_rows)
display(file_audit)
file_audit.to_csv(PROJECT_ROOT / 'results' / 'cicids2017_source_file_audit.csv', index=False)
print('Saved: cicids2017_source_file_audit.csv')


In [ ]:
# ============================================================
# 2. Reproduce the combined cleaning stage and reconcile counts
# ============================================================
data_raw = pd.concat(dfs, ignore_index=True)
data_raw.columns = data_raw.columns.str.strip()

raw_rows, raw_columns = data_raw.shape
missing_cells = int(data_raw.isna().sum().sum())
numeric_raw = data_raw.select_dtypes(include=[np.number])
infinite_cells = int(np.isinf(numeric_raw.to_numpy(dtype=np.float64, copy=False)).sum()) if numeric_raw.shape[1] else 0
missing_row = data_raw.isna().any(axis=1)
inf_row = pd.Series(
    np.isinf(numeric_raw.to_numpy(dtype=np.float64, copy=False)).any(axis=1),
    index=data_raw.index
) if numeric_raw.shape[1] else pd.Series(False, index=data_raw.index)
invalid_row = missing_row | inf_row

data_clean = data_raw.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

print('Merged raw shape:', data_raw.shape)
print('Missing cells before cleaning:', f'{missing_cells:,}')
print('Infinite numeric cells before cleaning:', f'{infinite_cells:,}')
print('Rows with any missing value:', f'{int(missing_row.sum()):,}')
print('Rows with any infinity:', f'{int(inf_row.sum()):,}')
print('Rows with both missing and infinity:', f'{int((missing_row & inf_row).sum()):,}')
print('Rows removed by the notebook cleaning rule:', f'{int(invalid_row.sum()):,}')
print('Combined shape after replace(inf -> NaN) + dropna:', data_clean.shape)
print('Sum of per-file retained rows:', f'{int(file_audit["Rows_retained_after_current_cleaning"].sum()):,}')
assert int(file_audit['Raw_rows'].sum()) == raw_rows, 'Per-file raw counts do not reconcile with merged raw count.'
assert int(file_audit['Rows_retained_after_current_cleaning'].sum()) == len(data_clean), 'Per-file retained counts do not reconcile with combined cleaning.'
print('✅ Per-file and combined row counts reconcile.')

raw_labels = data_clean[TARGET_COL].astype(str)
print('\nLabel values after current cleaning (preserving whitespace/case):')
raw_label_counts = raw_labels.value_counts(dropna=False).rename_axis('Raw_label').reset_index(name='Rows')
# Normalize the legacy 0x96 dash for display/export only; raw_labels remains unchanged for binary mapping.
raw_label_counts['Raw_label'] = raw_label_counts['Raw_label'].str.replace(chr(0x96), '–', regex=False)
display(raw_label_counts)
raw_label_counts.to_csv('cicids2017_labels_after_cleaning.csv', index=False)

# Reproduce the original notebook's exact binary mapping; separately flag labels
# whose trimmed representation differs from the raw stored label.
y_binary = raw_labels.apply(lambda value: 0 if value == 'BENIGN' else 1)
trimmed_labels = raw_labels.str.strip()
print('Exact BENIGN label rows:', int((raw_labels == 'BENIGN').sum()))
print('Whitespace-affected label rows:', int((raw_labels != trimmed_labels).sum()))
print('Trimmed BENIGN rows (diagnostic only):', int((trimmed_labels == 'BENIGN').sum()))
print('Binary class distribution using exact original mapping:')
display(y_binary.value_counts().sort_index().rename(index={0: 'Benign (0)', 1: 'Attack (1)'}).rename_axis('Class').reset_index(name='Rows'))


In [ ]:
# ============================================================
# 3. Reproduce constant-feature removal and duplicate removal
# ============================================================
X_all = data_clean.drop(columns=[TARGET_COL])
nunique = X_all.nunique(dropna=True)
constant_columns = nunique[nunique <= 1].index.tolist()
X_reduced = X_all.loc[:, nunique > 1].copy()

df_for_dedup = X_reduced.copy()
df_for_dedup[TARGET_COL] = y_binary.to_numpy()
rows_before_dedup = len(df_for_dedup)
duplicate_rows = int(df_for_dedup.duplicated().sum())
df_final = df_for_dedup.drop_duplicates().reset_index(drop=True)
rows_after_dedup = len(df_final)

print('Feature columns before constant removal (target excluded):', X_all.shape[1])
print('Constant columns removed:', len(constant_columns))
print('Constant column names:', constant_columns)
print('Columns remaining after constant removal (target excluded):', X_reduced.shape[1])
print('Rows before deduplication:', f'{rows_before_dedup:,}')
print('Exact duplicate rows removed:', f'{duplicate_rows:,}')
print('Rows after deduplication:', f'{rows_after_dedup:,}')
print('\nBinary labels after cleaning, before deduplication:')
display(pd.Series(y_binary).value_counts().sort_index().rename(index={0: 'Benign (0)', 1: 'Attack (1)'}).rename_axis('Class').reset_index(name='Rows'))
print('Binary labels after deduplication:')
display(df_final[TARGET_COL].value_counts().sort_index().rename(index={0: 'Benign (0)', 1: 'Attack (1)'}).rename_axis('Class').reset_index(name='Rows'))

feature_audit = pd.DataFrame({
    'Feature': X_reduced.columns,
    'Dtype_after_cleaning': X_reduced.dtypes.astype(str).values,
    'Unique_values_after_cleaning': [int(X_reduced[c].nunique(dropna=True)) for c in X_reduced.columns],
    'Used_by_numeric_only_pipeline': [pd.api.types.is_numeric_dtype(X_reduced[c]) for c in X_reduced.columns],
})
display(feature_audit)
feature_audit.to_csv(PROJECT_ROOT / 'results' / 'cicids2017_final_feature_audit.csv', index=False)
print('Saved: cicids2017_final_feature_audit.csv')

df_final[TARGET_COL].value_counts().sort_index().rename(index={0: 'Benign (0)', 1: 'Attack (1)'}).rename_axis('Class').reset_index(name='Rows').to_csv(
    PROJECT_ROOT / 'results' / 'cicids2017_labels_after_dedup.csv', index=False
)
print('Saved: cicids2017_labels_after_dedup.csv')


In [ ]:
# ============================================================
# 4. Audit the actual numeric feature dimension and data split
# ============================================================
X_final = df_final.drop(columns=[TARGET_COL])
y_final = df_final[TARGET_COL]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_final, y_final, test_size=0.2, stratify=y_final, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

numeric_train = X_train.select_dtypes(include=[np.number])
numeric_val = X_val.select_dtypes(include=[np.number])
numeric_test = X_test.select_dtypes(include=[np.number])
numeric_features = list(numeric_train.columns)

print('Train rows:', f'{len(X_train):,}')
print('Validation rows:', f'{len(X_val):,}')
print('Test rows:', f'{len(X_test):,}')
print('Columns remaining after constant removal:', X_final.shape[1])
print('Numeric features actually available to StandardScaler / AE:', len(numeric_features))
print('Non-numeric columns removed by select_dtypes:', [c for c in X_final.columns if c not in numeric_features])
print('Train/validation/test numeric feature counts:', numeric_train.shape[1], numeric_val.shape[1], numeric_test.shape[1])
assert list(numeric_val.columns) == numeric_features
assert list(numeric_test.columns) == numeric_features
assert len(X_train) + len(X_val) + len(X_test) == len(df_final)
print('✅ Split and numeric feature schemas align.')

summary = [
    ('Raw merged rows', raw_rows),
    ('Raw columns including label', raw_columns),
    ('Missing cells before cleaning', missing_cells),
    ('Infinite numeric cells before cleaning', infinite_cells),
    ('Rows dropped by current NaN/Inf cleaning rule', int(invalid_row.sum())),
    ('Rows retained after NaN/Inf cleaning', len(data_clean)),
    ('Feature columns before constant removal (target excluded)', X_all.shape[1]),
    ('Constant columns removed', len(constant_columns)),
    ('Columns remaining after constant removal', X_reduced.shape[1]),
    ('Duplicate rows removed', duplicate_rows),
    ('Final rows after duplicate removal', len(df_final)),
    ('Final benign rows', int((df_final[TARGET_COL] == 0).sum())),
    ('Final attack rows', int((df_final[TARGET_COL] == 1).sum())),
    ('Train rows', len(X_train)),
    ('Validation rows', len(X_val)),
    ('Test rows', len(X_test)),
    ('Numeric features actually used by AE', len(numeric_features)),
    ('Frozen manuscript claim: numeric features', 76),
]
summary_df = pd.DataFrame(summary, columns=['Audit_item', 'Observed_value'])
display(summary_df)
summary_df.to_csv(PROJECT_ROOT / 'results' / 'cicids2017_cleaning_summary.csv', index=False)
print('Saved: cicids2017_cleaning_summary.csv')

print('\nMANUSCRIPT CHECK:')
print(f'- Replace the current 76-D input claim with {len(numeric_features)}-D if that is the actual numeric count reported above.')
print(f'- Explain {int(invalid_row.sum()):,} removed rows using the per-file audit; do not describe them as duplicates.')
print(f'- Use the post-dedup class counts printed above for the final-data description.')


## What to send back

Send screenshots or upload these generated CSV files:
- `cicids2017_source_file_audit.csv`
- `cicids2017_cleaning_summary.csv`
- `cicids2017_final_feature_audit.csv`
- `cicids2017_labels_after_dedup.csv`

These reports will let us revise the dataset description, feature dimension (if needed), and class counts in the manuscript without inventing values.
